## 🌾 Comprehensive Haystack Tutorial

This notebook provides an exhaustive guide to the **Haystack** framework by deepset, an open-source tool for building LLM-powered applications like retrieval-augmented generation (RAG), question answering, semantic search, and more. It expands on previous versions by adding advanced functionalities such as hybrid retrieval, conditional routing, document classification, web RAG, LangChain integration, advanced document processing, and multi-modal RAG.

## What is Haystack?
Haystack enables customizable AI applications through modular pipelines, connecting components like retrievers, generators, and embedders. It integrates with Hugging Face, OpenAI, Weaviate, Chroma, and other tools for production-grade NLP systems.

## Objectives
- Cover basic to advanced Haystack functionalities.
- Provide practical examples with theoretical context.
- Ensure JSON validity to prevent parsing errors.

## Prerequisites
- Python 3.10+
- Install dependencies: `pip install haystack-ai haystack-integrations[chroma] openai sentence-transformers weaviate-client langchain requests transformers torch torchvision`
- Sample data directory (`data/`) with `sample.txt` (any text file) and `sample_image.jpg` (any image file).
- Optional: OpenAI API key, Weaviate instance, and internet access for web RAG.

## Structure
1. Setup and Basic RAG Pipeline
2. Document Store and Data Ingestion
3. Semantic Search
4. Extractive Question Answering
5. Custom Prompt Templates
6. Advanced RAG with Ranking
7. Agentic Pipeline with Function Calling
8. Evaluation and Metrics
9. Hybrid Retrieval (Dense + Sparse)
10. Conditional Routing in Pipelines
11. Document Classification
12. Web RAG with WebRetriever
13. Integration with LangChain
14. Deployment with Hayhooks
15. Advanced Document Processing
16. Multi-Modal RAG

Let's dive in!

## 1. Setup and Basic RAG Pipeline

**Theory**: Haystack pipelines combine retrievers and generators for RAG, simplifying complex NLP tasks.

In [ ]:
# Install required packages
!pip install haystack-ai haystack-integrations[chroma] openai sentence-transformers weaviate-client langchain requests transformers torch torchvision

In [ ]:
# Basic RAG Pipeline
from haystack import Pipeline
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.generators import OpenAIGenerator
from haystack.components.builders import PromptBuilder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document

# Initialize document store
document_store = InMemoryDocumentStore()
documents = [Document(content="Haystack is an open-source framework for LLM applications.")]
document_store.write_documents(documents)

# Define prompt template
prompt_template = """
Given the following documents:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}
Answer the question: {{ query }}
Answer:
"""

# Create pipeline
pipeline = Pipeline()
pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipeline.add_component("prompt_builder", PromptBuilder(template=prompt_template))
pipeline.add_component("generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.connect("retriever.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "generator.prompt")

# Run pipeline
query = "What is Haystack?"
result = pipeline.run({"retriever": {"query": query}, "prompt_builder": {"query": query}})
print(f"Question: {query}")
print(f"Answer: {result['generator']['replies'][0]}")

## 2. Document Store and Data Ingestion

**Theory**: Document stores manage data for retrieval, supporting vector databases like Chroma.

In [ ]:
# Data Ingestion with Chroma
from haystack_integrations.document_stores.chroma import ChromaDocumentStore
from haystack.components.converters import TextFileToDocument
from haystack.components.preprocessors import DocumentSplitter
from haystack import Pipeline

# Initialize Chroma document store
document_store = ChromaDocumentStore(collection_name="documents", persist_path=".\\chroma_db")

# Create ingestion pipeline
ingestion_pipeline = Pipeline()
ingestion_pipeline.add_component("converter", TextFileToDocument())
ingestion_pipeline.add_component("splitter", DocumentSplitter(split_by="sentence", split_length=2))
ingestion_pipeline.add_component("store", document_store)
ingestion_pipeline.connect("converter.documents", "splitter.documents")
ingestion_pipeline.connect("splitter.documents", "store.documents")

# Ingest sample text file
result = ingestion_pipeline.run({"converter": {"sources": ["data\\sample.txt"]}})
print(f"Documents ingested: {document_store.count_documents()}")

## 3. Semantic Search

**Theory**: Semantic search retrieves documents based on meaning using embeddings.

In [ ]:
# Semantic Search with Chroma
from haystack_integrations.components.retrievers.chroma import ChromaEmbeddingRetriever
from haystack import Pipeline

# Create search pipeline
search_pipeline = Pipeline()
search_pipeline.add_component("retriever", ChromaEmbeddingRetriever(document_store=document_store))

# Run search
query = "AI frameworks"
result = search_pipeline.run({"retriever": {"query": query, "top_k": 2}})
for doc in result["retriever"]["documents"]:
    print(f"Document: {doc.content}\nScore: {doc.score:.4f}\n")

## 4. Extractive Question Answering

**Theory**: Extractive QA identifies answers within documents.

In [ ]:
# Extractive Question Answering
from haystack.components.readers import ExtractiveReader
from haystack import Pipeline

# Create QA pipeline
qa_pipeline = Pipeline()
qa_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
qa_pipeline.add_component("reader", ExtractiveReader())
qa_pipeline.connect("retriever.documents", "reader.documents")

# Run QA
query = "What is Haystack used for?"
result = qa_pipeline.run({"retriever": {"query": query}, "reader": {"query": query}})
for answer in result["reader"]["answers"]:
    print(f"Answer: {answer.text}\nScore: {answer.score:.4f}\n")

## 5. Custom Prompt Templates

**Theory**: Custom prompts enhance LLM interactions.

In [ ]:
# Custom Prompt Template
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage

# Define custom template
prompt_template = [
    ChatMessage.from_user(
        "Summarize the following documents in one sentence:\n{% for doc in documents %}{{ doc.content }}\n{% endfor %}"
    )
]

# Create pipeline
pipeline = Pipeline()
pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipeline.add_component("prompt_builder", ChatPromptBuilder(template=prompt_template))
pipeline.add_component("generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.connect("retriever.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "generator.messages")

# Run pipeline
result = pipeline.run({"retriever": {"query": "Haystack features"}})
print(f"Summary: {result['generator']['replies'][0].text}")

## 6. Advanced RAG with Ranking

**Theory**: Ranking refines retrieved documents for better accuracy.

In [ ]:
# Advanced RAG with Ranking
from haystack.components.rankers import TransformersSimilarityRanker
from haystack import Pipeline

# Create pipeline
pipeline = Pipeline()
pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=document_store, top_k=5))
pipeline.add_component("ranker", TransformersSimilarityRanker(top_k=2))
pipeline.add_component("prompt_builder", PromptBuilder(template=prompt_template))
pipeline.add_component("generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.connect("retriever.documents", "ranker.documents")
pipeline.connect("ranker.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "generator.prompt")

# Run pipeline
query = "Haystack capabilities"
result = pipeline.run({"retriever": {"query": query}, "ranker": {"query": query}, "prompt_builder": {"query": query}})
print(f"Answer: {result['generator']['replies'][0]}")

## 7. Agentic Pipeline with Function Calling

**Theory**: Agentic pipelines enable complex workflows with tool use.

In [ ]:
# Agentic Pipeline
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage
from haystack import Pipeline

# Define pipeline
pipeline = Pipeline()
pipeline.add_component("generator", OpenAIChatGenerator(api_key="your_openai_api_key"))

# Run agentic query
messages = [ChatMessage.from_user("Search for AI frameworks and summarize findings.")]
result = pipeline.run({"generator": {"messages": messages}})
print(f"Agentic Response: {result['generator']['replies'][0].text}")

## 8. Evaluation and Metrics

**Theory**: Haystack supports pipeline evaluation with metrics like recall.

In [ ]:
# Evaluation
from haystack.components.evaluators import DocumentRecallEvaluator
from haystack import Pipeline

# Create evaluation pipeline
eval_pipeline = Pipeline()
eval_pipeline.add_component("evaluator", DocumentRecallEvaluator())

# Run evaluation
result = eval_pipeline.run({
    "evaluator": {
        "documents": [Document(content="Haystack is great.")],
        "ground_truth_documents": [Document(content="Haystack is great.")]
    }
})
print(f"Recall: {result['evaluator']['recall']}")

## 9. Hybrid Retrieval (Dense + Sparse)

**Theory**: Hybrid retrieval combines dense (embedding-based) and sparse (keyword-based) methods for improved accuracy.

In [ ]:
# Hybrid Retrieval
from haystack_integrations.components.retrievers.chroma import ChromaEmbeddingRetriever
from haystack.components.joiners import DocumentJoiner
from haystack import Pipeline

# Create hybrid pipeline
pipeline = Pipeline()
pipeline.add_component("bm25_retriever", InMemoryBM25Retriever(document_store=document_store, top_k=5))
pipeline.add_component("dense_retriever", ChromaEmbeddingRetriever(document_store=document_store, top_k=5))
pipeline.add_component("joiner", DocumentJoiner())
pipeline.add_component("prompt_builder", PromptBuilder(template=prompt_template))
pipeline.add_component("generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.connect("bm25_retriever.documents", "joiner.documents")
pipeline.connect("dense_retriever.documents", "joiner.documents")
pipeline.connect("joiner.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "generator.prompt")

# Run pipeline
query = "AI framework features"
result = pipeline.run({
    "bm25_retriever": {"query": query},
    "dense_retriever": {"query": query},
    "prompt_builder": {"query": query}
})
print(f"Hybrid Answer: {result['generator']['replies'][0]}")

## 10. Conditional Routing in Pipelines

**Theory**: Conditional routing directs pipeline flow based on conditions, e.g., query type.

In [ ]:
# Conditional Routing
from haystack.components.routers import ConditionalRouter
from haystack import Pipeline

# Define routing conditions
routes = [
    {
        "condition": "{{ query | lower | contains('summary') }}",
        "output": "summary",
        "output_name": "route"
    },
    {
        "condition": "{{ query | lower | contains('question') }}",
        "output": "qa",
        "output_name": "route"
    }
]

# Create pipeline
pipeline = Pipeline()
pipeline.add_component("router", ConditionalRouter(routes))
pipeline.add_component("summary_generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.add_component("qa_retriever", InMemoryBM25Retriever(document_store=document_store))
pipeline.connect("router.summary", "summary_generator.prompt")
pipeline.connect("router.qa", "qa_retriever.query")

# Run pipeline
query = "Summarize AI frameworks"
result = pipeline.run({"router": {"query": query}})
if "summary_generator" in result:
    print(f"Summary: {result['summary_generator']['replies'][0]}")
else:
    print(f"Retrieved Documents: {len(result['qa_retriever']['documents'])}")

## 11. Document Classification

**Theory**: Haystack can classify documents (e.g., sentiment analysis) using transformers.

In [ ]:
# Document Classification
from haystack.components.classifiers import DocumentClassifier
from haystack import Pipeline

# Create pipeline
pipeline = Pipeline()
pipeline.add_component("classifier", DocumentClassifier(model_name_or_path="distilbert-base-uncased-finetuned-sst-2-english"))

# Run classification
documents = [Document(content="Haystack is amazing!"), Document(content="This is terrible.")]
result = pipeline.run({"classifier": {"documents": documents}})
for doc in result["classifier"]["documents"]:
    print(f"Document: {doc.content}\nLabel: {doc.meta['classification']['label']}\nScore: {doc.meta['classification']['score']:.4f}\n")

## 12. Web RAG with WebRetriever

**Theory**: WebRetriever fetches online content for RAG, enabling real-time knowledge.

In [ ]:
# Web RAG with WebRetriever
from haystack.components.fetchers import LinkContentFetcher
from haystack.components.converters import HTMLToDocument
from haystack import Pipeline

# Create pipeline
pipeline = Pipeline()
pipeline.add_component("fetcher", LinkContentFetcher())
pipeline.add_component("converter", HTMLToDocument())
pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipeline.add_component("prompt_builder", PromptBuilder(template=prompt_template))
pipeline.add_component("generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.connect("fetcher.streams", "converter.sources")
pipeline.connect("converter.documents", "retriever.documents")
pipeline.connect("retriever.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "generator.prompt")

# Run pipeline
query = "What are recent AI trends?"
result = pipeline.run({
    "fetcher": {"urls": ["https://www.deepset.ai/blog"]},
    "retriever": {"query": query},
    "prompt_builder": {"query": query}
})
print(f"Web RAG Answer: {result['generator']['replies'][0]}")

## 13. Integration with LangChain

**Theory**: Haystack pipelines can be used within LangChain for enhanced RAG.

In [ ]:
# Integration with LangChain
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from haystack_integrations.document_stores.chroma import ChromaDocumentStore
from haystack_integrations.components.retrievers.chroma import ChromaEmbeddingRetriever

# Initialize Haystack document store and retriever
document_store = ChromaDocumentStore(collection_name="langchain", persist_path=".\\chroma_langchain")
retriever = ChromaEmbeddingRetriever(document_store=document_store)

# Create LangChain chain
llm = OpenAI(api_key="your_openai_api_key")
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever.as_langchain_retriever())

# Run query
query = "What is Haystack?"
result = qa_chain.run(query)
print(f"LangChain Answer: {result}")

## 14. Deployment with Hayhooks

**Theory**: Hayhooks enables REST API deployment of pipelines.

In [ ]:
# Deployment with Hayhooks (Conceptual)
# Note: Requires running `hayhooks` server separately
print("To deploy, save pipeline as YAML and use Hayhooks:\n1. Export pipeline: pipeline.to_yaml('rag_pipeline.yaml')\n2. Run: hayhooks rag_pipeline.yaml\n3. Access API at http://localhost:1416")

## 15. Advanced Document Processing

**Theory**: Advanced preprocessing like cleaning, entity extraction, and metadata enrichment improves document quality for retrieval.

In [ ]:
# Advanced Document Processing
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.converters import TextFileToDocument
from haystack import Pipeline, Document
from transformers import pipeline as hf_pipeline

# Initialize NER pipeline for entity extraction
ner_pipeline = hf_pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english")

# Create processing pipeline
processing_pipeline = Pipeline()
processing_pipeline.add_component("converter", TextFileToDocument())
processing_pipeline.add_component("cleaner", DocumentCleaner(remove_empty_lines=True, remove_extra_whitespace=True))
processing_pipeline.add_component("splitter", DocumentSplitter(split_by="sentence", split_length=2))
processing_pipeline.add_component("store", document_store)
processing_pipeline.connect("converter.documents", "cleaner.documents")
processing_pipeline.connect("cleaner.documents", "splitter.documents")
processing_pipeline.connect("splitter.documents", "store.documents")

# Custom function for entity extraction and metadata enrichment
def enrich_with_entities(documents):
    for doc in documents:
        entities = ner_pipeline(doc.content)
        doc.meta["entities"] = [ent['word'] for ent in entities if ent['score'] > 0.8]
    return documents

# Run pipeline and enrich
result = processing_pipeline.run({"converter": {"sources": ["data\\sample.txt"]}})
documents = result["store"]["documents"]
enriched_docs = enrich_with_entities(documents)
for doc in enriched_docs:
    print(f"Document: {doc.content}\nEntities: {doc.meta.get('entities', [])}\n")

## 16. Multi-Modal RAG

**Theory**: Multi-modal RAG combines text and image data for richer context, using vision-language models.

In [ ]:
# Multi-Modal RAG (Conceptual with Text and Image)
from haystack import Pipeline, Document
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from torchvision import models, transforms
from PIL import Image

# Load pre-trained ResNet for image feature extraction
resnet = models.resnet50(pretrained=True)
resnet.eval()
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Process image
image = Image.open("data\\sample_image.jpg")
image_tensor = transform(image).unsqueeze(0)
image_description = "Image-based description (replace with vision-language model output)."

# Create multi-modal pipeline
pipeline = Pipeline()
pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipeline.add_component("prompt_builder", PromptBuilder(template=prompt_template))
pipeline.add_component("generator", OpenAIGenerator(api_key="your_openai_api_key"))
pipeline.connect("retriever.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "generator.prompt")

# Add image description to documents
document_store.write_documents([Document(content=image_description)])

# Run pipeline
query = "Describe the content including images"
result = pipeline.run({"retriever": {"query": query}, "prompt_builder": {"query": query}})
print(f"Multi-Modal Answer: {result['generator']['replies'][0]}")

## Conclusion

This comprehensive notebook covers Haystack’s core and advanced functionalities, from RAG to multi-modal retrieval and advanced document processing. Explore more at [Haystack Documentation](https://docs.haystack.deepset.ai/).